In [ ]:
# load libraries and data
import matplotlib.pyplot as plt
from scipy.io import loadmat
import numpy as np
from scipy.signal import welch
from scipy.signal import periodogram
import os
from scipy.signal import find_peaks, peak_prominences
from sklearn.linear_model import LinearRegression
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
from itertools import combinations
from tslearn.clustering import TimeSeriesKMeans
from tslearn.metrics import dtw, cdist_dtw
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score



# Define the list of experiments - 944dp
experiments = ['BD30-HD1-Asand', 'BD30-HD1-defaultsand', 'BD30-HD2-Asand', 'BD30-HD2-Cement', 'BD30-HD2-defaultsand', 'BD30-HD2-Msand', 
               'BD30-HD3-Asand', 'BD30-HD3-defaultsand', 'BD30-HD3-Msand', 'BD30-HD4-Asand', 'BD30-HD4-Cement', 'BD30-HD4-defaultsand', 
               'BD30-HD4-Msand']
# Define the trials for each experiment
trials = {'BD30-HD1-Asand':[1], 'BD30-HD1-defaultsand':[2], 'BD30-HD2-Asand':[1, 2], 'BD30-HD2-Cement':[1], 'BD30-HD2-defaultsand':[1, 2], 
          'BD30-HD2-Msand':[1, 2], 'BD30-HD3-Asand':[1, 2, 3, 4, 5], 'BD30-HD3-defaultsand':[3, 4, 6, 7], 'BD30-HD3-Msand':[1, 2, 3], 
          'BD30-HD4-Asand':[1], 'BD30-HD4-Cement':[1], 'BD30-HD4-defaultsand':[1, 2, 3], 'BD30-HD4-Msand':[1]}
# Define the number of runs for each trial in each experiment
runs = {'BD30-HD1-Asand': {1: 32}, 'BD30-HD1-defaultsand': {2: 51}, 'BD30-HD2-Asand': {1: 37, 2: 33}, 'BD30-HD2-Cement': {1: 39}, 
        'BD30-HD2-defaultsand': {1: 37, 2: 50}, 'BD30-HD2-Msand': {1: 39, 2: 58}, 'BD30-HD3-Asand': {1: 34, 2: 41, 3: 29, 4: 27, 5: 41}, 
        'BD30-HD3-defaultsand': {3: 10, 4: 14, 6:13, 7:52}, 'BD30-HD3-Msand': {1: 27, 2: 32, 3: 57}, 'BD30-HD4-Asand': {1: 30}, 
        'BD30-HD4-Cement': {1: 20}, 'BD30-HD4-defaultsand': {1: 32, 2: 31, 3: 40}, 'BD30-HD4-Msand': {1: 38}}


total_runs = 0
for exp in experiments:
    for tr in trials[exp]:
        total_runs += runs[exp][tr]

print('The total number of runs included in the clustering is: {}'.format(total_runs))


# define the data_for_clustering array size based on which peridoogram smoothing you choose here
# for a welch (psd_type = [65, 129, 257, 513]) 
psd_type=129
data_for_clustering = np.zeros((total_runs, psd_type))

# OR instead for original periodogram
#frames_chosen = 1800
#f, powers = periodogram(list(range(frames_chosen)), fs=124.5)        
#data_for_clustering = np.zeros((total_runs,len(powers)))

# directory of the matlab files github version
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()


# Extract the data_for_clustering
count = 0
for exp in experiments:
    for tr in trials[exp]:
        trial_dir = os.path.join(base_dir, f"{exp}-Trial{tr}")
        for r in range(runs[exp][tr]):

            #exporting experimental data from matlab 
            folder_name = 'P' + str(r+1)
            file_name = os.path.join(trial_dir, folder_name, f"{folder_name}.mat")
            if not os.path.exists(file_name):
                raise FileNotFoundError(f"Missing file:\n{file_name}\n\n"f"Expected extracted data folder next to the code:\n{trial_dir}")
            
            all_mat_data = loadmat(file_name)
            image = all_mat_data['ImagingData']['Image'][0][0]
            image_clipped = np.clip(image, 0, 1)    
            
            #extracting the periodogram data
            data = image_clipped[:1800, :, :, :]
            
            # for a welch configuration
            pows = np.zeros(psd_type)
            # OR instead for original periodogram
            #f, powers = periodogram(data[:, 10, 10, 10], fs=124.5)
            #pows = np.zeros(len(powers)) #or len(freqs)
            
            for x in range(20):
                for y in range (20):
                    for z in range(20):
                        if (
                        ((y==0 or y==19) and (x<=6 or x>=13)) or 
                        ((y==1 or y==18) and (x<=4 or x>=15)) or
                        ((y==2 or y==17) and (x<=2 or x>=17)) or 
                        ((y==3 or y==16 or y==4 or y==15) and (x<=1 or x>=18)) or 
                        ((y==5 or y==14 or y==6 or y==13) and (x==0 or x==19))
                        )==False:
                            # for a welch (in this case 129PSD)
                            f, p = welch(data[:, x, y, z], fs=124.5, nperseg=2 * (psd_type - 1))
                            # OR instead for original periodogram
                            #f, p = periodogram(data[:, x, y, z], fs=124.5)

                            pows += p
            data_for_clustering[count,:] = pows.T 
            count += 1

log_data = np.log10(data_for_clustering[:, 4:])
freqs = f[4:]
log_data_ts = log_data[:, :, None]

In [ ]:
# kmeans
# ---------------------------
# PARAMETERS
# ---------------------------
n_clusters = 4
required_runs = 100
target_family_size = 3
dtw_similarity_threshold = 3.5
hamming_threshold = 0.05

log_data = np.log10(data_for_clustering[:, 4:])
log_data_ts = log_data[:, :, None]
freqs = f[4:]
D = cdist_dtw(log_data_ts)

# ---------------------------
# HELPERS
# ---------------------------
def reorder_mean_sd(centroids, labels):
    means = np.array([c.mean() for c in centroids])
    stds = np.array([c.std() for c in centroids])

    mean_order = np.argsort(means)

    if len(mean_order) <= 2:
        order = mean_order
    else:
        first = mean_order[0]
        last = mean_order[-1]
        middle = mean_order[1:-1]
        middle = middle[np.argsort(-stds[middle])]  
        order = np.r_[first, middle, last]

    centroids = centroids[order]
    new_labels = np.zeros_like(labels)
    for new, old in enumerate(order):
        new_labels[labels == old] = new
    return centroids, new_labels

def hamming(a, b):
    return np.mean(np.asarray(a) != np.asarray(b))

def plot_centroids(freqs, centroids, title):
    plt.figure(figsize=(8, 5))
    for i in range(len(centroids)):
        plt.plot(freqs, centroids[i].ravel(), label=f'Cluster {i}')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Log(PSD)')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

def best_clique_by_silhouette(H, size, threshold, silhouettes):
    if H.shape[0] < size:
        return None
    valid = []
    for combo in combinations(range(H.shape[0]), size):
        if all(H[i, j] < threshold for i, j in combinations(combo, 2)):
            valid.append(combo)
    if not valid:
        return None
    return max(valid, key=lambda combo: np.mean([silhouettes[i] for i in combo]))

# ---------------------------
# STORAGE
# ---------------------------
labels_list = []
centroids_list = []
silhouette_list = []
db_list = []
ch_list = []
run_indices = []
num_rejected_by_centroid_similarity = 0

H = np.empty((0, 0))
best = None
selection_mode = None

# ---------------------------
# MAIN LOOP
# ---------------------------
for run_idx in range(required_runs):
    model = TimeSeriesKMeans(n_clusters=n_clusters, metric='dtw', max_iter=100)
    model.fit(log_data_ts)

    centroids = model.cluster_centers_
    labels = model.labels_

    # centroid DTW-separation filter
    d = [dtw(centroids[i], centroids[j]) for i, j in combinations(range(n_clusters), 2)]
    if np.any(np.array(d) < dtw_similarity_threshold):
        num_rejected_by_centroid_similarity += 1
        print(f"Run {run_idx}: rejected by centroid DTW threshold")
        continue

    # deterministic reordering
    centroids, labels = reorder_mean_sd(centroids, labels)
    sil = silhouette_score(D, labels, metric="precomputed")
    db = davies_bouldin_score(log_data, labels)
    ch = calinski_harabasz_score(log_data, labels)

    print(f"\nRun {run_idx}: accepted")
    print(f"Silhouette = {sil:.4f}")
    print(f"DB = {db:.4f}")
    print(f"CH = {ch:.4f}")
    print("First 200 reordered labels:")
    print(labels[:200])
    plot_centroids(freqs, centroids, f"Accepted run {run_idx}")

    # store accepted run
    labels_list.append(labels.copy())
    centroids_list.append(centroids.copy())
    silhouette_list.append(sil)
    db_list.append(db)
    ch_list.append(ch)
    run_indices.append(run_idx)

    # expand Hamming matrix
    m_old = H.shape[0]
    m_new = m_old + 1
    H_new = np.full((m_new, m_new), np.nan)
    if m_old > 0:
        H_new[:m_old, :m_old] = H
    H_new[m_old, m_old] = 0.0

    for j in range(m_old):
        hij = hamming(labels_list[m_old], labels_list[j])
        H_new[m_old, j] = hij
        H_new[j, m_old] = hij

    H = H_new

    # try to find a stable family using all currently accepted runs
    final_size = min(target_family_size, len(labels_list))
    if final_size >= 2:
        best = best_clique_by_silhouette(H, final_size, hamming_threshold, silhouette_list)
        if best is not None and len(best) == target_family_size:
            selection_mode = "stable family"
            print("\nStable family found.")
            print("Local indices:", best)
            print("Original run indices:", [run_indices[i] for i in best])
            break

# ---------------------------
# FINAL SELECTION
# ---------------------------
final_size = min(target_family_size, len(labels_list))

if final_size == 0:
    best = None
    selection_mode = "no accepted runs"
else:
    stable_best = best_clique_by_silhouette(H, final_size, hamming_threshold, silhouette_list)
    if stable_best is not None:
        best = stable_best
        selection_mode = f"stable family of {final_size}"
    else:
        best = tuple(np.argsort(silhouette_list)[-final_size:][::-1])
        selection_mode = f"fallback: top {final_size} accepted by silhouette"

# ---------------------------
# FINAL REPORT
# ---------------------------
if best is None:
    print("\nNo accepted runs found.")
else:
    print(f"\nFINAL SELECTED SET ({selection_mode})")
    print("Local indices:", best)
    print("Original run indices:", [run_indices[i] for i in best])

    if len(best) >= 2:
        print("\nPairwise Hamming matrix for selected set:")
        selected_H = H[np.ix_(best, best)]
        print(np.round(selected_H, 4))

    for k, idx in enumerate(best, 1):
        print(f"\nSelected run {k} | original run = {run_indices[idx]}")
        print(f"Silhouette = {silhouette_list[idx]:.4f}")
        print(f"DB = {db_list[idx]:.4f}")
        print(f"CH = {ch_list[idx]:.4f}")
        print("First 200 reordered labels:")
        print(labels_list[idx][:200])
        plot_centroids(freqs, centroids_list[idx], f"Selected run {run_indices[idx]}")

    print("\nAverage metrics across selected set:")
    print(f"Silhouette = {np.mean([silhouette_list[i] for i in best]):.4f}")
    print(f"DB = {np.mean([db_list[i] for i in best]):.4f}")
    print(f"CH = {np.mean([ch_list[i] for i in best]):.4f}")
    if len(best) >= 2:
        pair_vals = [H[i, j] for i, j in combinations(best, 2)]
        print(f"Average pairwise Hamming = {np.mean(pair_vals):.4f}")

# ---------------------------
# SUMMARY
# ---------------------------
total_checked = run_idx + 1
accepted_after_dtw = len(labels_list)

print(f"\nAccepted runs stored = {accepted_after_dtw}")
print(f"Rejected by centroid DTW threshold = {num_rejected_by_centroid_similarity}")
print(f"Total clustering repetitions checked = {total_checked}")

similarity_ratio = num_rejected_by_centroid_similarity / total_checked
dissimilarity_ratio = accepted_after_dtw / total_checked

print(f"Similarity ratio = {num_rejected_by_centroid_similarity}/{total_checked} = {similarity_ratio:.4f}")
print(f"Dissimilarity ratio = {accepted_after_dtw}/{total_checked} = {dissimilarity_ratio:.4f}")

In [ ]:
# Regime map
labels = labels_list[-1]
centroids = centroids_list[-1]
centroids_reshaped = centroids[:, :, 0]   # (n_clusters, len(freqs))

areas_under_curve, std_devs, slopes = [], [], []

for i, per in enumerate(log_data):
    # AUC
    auc = np.trapz(per, freqs)
    areas_under_curve.append(auc)

    # Standard Deviation/Mean (S/M)
    std_dev = np.std(per) / np.mean(per)
    log_std_dev = np.log(np.abs(std_dev) + 1e-8) 
    std_devs.append(log_std_dev)

    # Slope (fit a linear regression to the curve)
    x = freqs.reshape(-1, 1)
    model = LinearRegression().fit(x, per)
    slope = model.coef_[0]  
    log_slope = np.log(np.abs(slope) + 1e-8)
    slopes.append(log_slope)



# Initialize lists to store the centroid values
centroid_areas_under_curve, centroid_std_devs, centroid_slopes = [], [], []

# Calculate AUC, S/M, and slope for each centroid
for i, centroid in enumerate(centroids_reshaped):
    centroid_auc = np.trapz(centroid, freqs)
    centroid_areas_under_curve.append(centroid_auc)

    centroid_std_dev = np.std(centroid) / np.mean(centroid)
    log_centroid_std_dev = np.log(np.abs(centroid_std_dev) + 1e-8)
    centroid_std_devs.append(log_centroid_std_dev)

    x = freqs.reshape(-1, 1)
    model = LinearRegression().fit(x, centroid)
    centroid_slope = model.coef_[0]
    log_centroid_slope = np.log(np.abs(centroid_slope) + 1e-8) 
    centroid_slopes.append(log_centroid_slope)

# Create a 3D scatter plot of AUC, log-transformed S/M, and log-transformed Slope
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(areas_under_curve, std_devs, slopes, c=labels, cmap='viridis', label='Periodogram Data', s=50)
ax.scatter(centroid_areas_under_curve, centroid_std_devs, centroid_slopes, 
           c=np.arange(len(centroids_reshaped)), cmap='viridis', marker='x', s=200, label='Centroids')
ax.set_xlabel('Area Under Curve (AUC)')
ax.set_ylabel('Log(Standard Deviation/Mean)')
ax.set_zlabel('Log(Slope)')
ax.set_title('3D Data Categorization based on AUC, Log(SD/M) and Log(Slope)')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Cluster Label')
plt.legend()
plt.show()

In [ ]:
# elbow method for timeserieskmeans
from kneed import KneeLocator

required_runs = 10
log_data = np.log10(data_for_clustering[:, 4:])
log_data_ts = log_data[:, :, None]
freqs = f[4:]
cluster_range = range(2, 11)
wcss_runs = []  

for n_clusters in cluster_range:
    print(f"\nStarting clustering for {n_clusters} clusters...")
    max_min_dtw_distance = -np.inf  # Initialize to track the highest minimum DTW distance
    best_centroids = None
    best_inertia = None
    run_idx = 0

    while run_idx < required_runs and max_min_dtw_distance<3.5:
        # Perform clustering
        model = TimeSeriesKMeans(n_clusters=n_clusters, metric='dtw', max_iter=100)
        model.fit(log_data_ts)
        centroids = model.cluster_centers_

        # Calculate DTW distances between centroids
        num_comparisons = n_clusters * (n_clusters - 1) // 2
        dtw_distances = np.zeros(num_comparisons)
        k = 0
        for i in range(n_clusters):
            for j in range(i + 1, n_clusters):
                dtw_distances[k] = dtw(centroids[i,:,0], centroids[j,:,0])
                k += 1

        # Find the minimum DTW distance for this run
        min_dtw_distance = np.min(dtw_distances)

        # Update if this run's minimum DTW distance is the largest found so far
        if min_dtw_distance > max_min_dtw_distance:
            max_min_dtw_distance = min_dtw_distance
            print(max_min_dtw_distance)
            best_centroids = centroids
            best_inertia = model.inertia_
        
        run_idx += 1
        print(run_idx)

    # Append the best inertia (WCSS) for this number of clusters
    wcss_runs.append(best_inertia)

    # Plot the best centroids for the current number of clusters
    plt.figure(figsize=(12, 10))
    for yi in range(n_clusters):
        plt.subplot(n_clusters, 1, 1 + yi)
        plt.plot(freqs, best_centroids[yi,:,0], label=f'Cluster {yi}')
        plt.ylim(-3.5, 1.5)
        plt.xlabel('Frequency')
        plt.ylabel('Log(PSD)')
        plt.title(f'Cluster {yi}')
        plt.legend()
    plt.tight_layout()
    plt.show()

    
cr = list(cluster_range)

# --- KneeLocator elbow ---
kl = KneeLocator(cr, wcss_runs, curve="convex", direction="decreasing")
elbow = kl.elbow

# --- Alternative elbow (percent-drop heuristic) ---
wcss_diff = np.diff(wcss_runs)
wcss_diff_ratio = np.abs(wcss_diff) / np.array(wcss_runs[:-1]) * 100

optimal_clusters = None
for i in range(1, len(wcss_diff_ratio)):
    if wcss_diff_ratio[i] < 10:   # your threshold
        optimal_clusters = cr[i + 1]
        break

if optimal_clusters is None:
    optimal_clusters = cr[-1]  

# --- Single combined plot ---
plt.plot(cr, wcss_runs, marker='o')
plt.title('Elbow Method (TimeSeriesKMeans)')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')

if elbow is not None:
    idx_elbow = cr.index(elbow)
    plt.axvline(x=elbow, linestyle='--', label=f'Kneedle elbow: {elbow}')
    plt.scatter(elbow, wcss_runs[idx_elbow], s=100, zorder=5)

idx_opt = cr.index(optimal_clusters)
plt.axvline(x=optimal_clusters, linestyle='--', label=f'%drop elbow: {optimal_clusters}')
plt.scatter(optimal_clusters, wcss_runs[idx_opt], s=100, zorder=5)

plt.legend()
plt.show()

print(f"Kneedle elbow: {elbow}")
print(f"%drop elbow (<10%): {optimal_clusters}")

In [ ]:
# KMedoids clustering algorithm
from tslearn.metrics import cdist_dtw
from sklearn_extra.cluster import KMedoids
from tslearn.metrics import dtw
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from tslearn.clustering import silhouette_score as ts_silhouette
import matplotlib.pyplot as plt


n_clusters = 4
log_data = np.log10(data_for_clustering[:, 4:])
freqs = f[4:]


model_KMedoids = KMedoids(n_clusters=n_clusters, method='pam', metric = 'euclidean', max_iter=100)
model_KMedoids.fit((log_data))
centroids_KMedoids = model_KMedoids.cluster_centers_
labels_KMedoids = model_KMedoids.labels_


# Plot medoid cluster centers as one plot
plt.figure(figsize=(10, 6))
for i, j in enumerate(centroids_KMedoids):
    plt.plot(freqs, j, label=f'Cluster {i} Center')
plt.xlabel("Frequency (Hz)")
plt.ylabel("Log(PSD)")
plt.title("Cluster Centers (KMedoids)")
plt.legend()
plt.show()

print("The array is divided as follows on the clusters: \n {}".format(labels_KMedoids))


num_comparisons = n_clusters * (n_clusters - 1) // 2
dtw_distances = np.zeros(num_comparisons)

k = 0
for i in range(n_clusters):
    for j in range(i + 1, n_clusters):
        dtw_distances[k] = dtw(centroids_KMedoids[i], centroids_KMedoids[j])
        print(f"DTW distance between centroid {i} and centroid {j}: {dtw_distances[k]:.4f}")
        k += 1

dtw_similarity_threshold = 3.5
simi=np.any(dtw_distances < dtw_similarity_threshold)
if simi:
    print(f"Warning: Some centroids are too similar in shape (DTW distance < {dtw_similarity_threshold}).")
else:
    print("Centroids are sufficiently different based on DTW distance.")


sil = ts_silhouette(log_data, labels_KMedoids)
print(f"Silhouette Score (KMedoids): {sil: .2f}")
db_index = davies_bouldin_score(log_data, labels_KMedoids)
print(f"Davies-Bouldin Index (KMedoids): {db_index: .2f}")
ch_score = calinski_harabasz_score(log_data, labels_KMedoids)
print(f"Calinski-Harabasz Index (KMedoids): {ch_score: .2f}")

# kmedoids elbow
wcss_KMedoids = []
for i in range(2, 11):
    clusters_KMedoids = KMedoids(n_clusters=i, method='pam', metric = 'euclidean', max_iter=100)
    clusters_KMedoids.fit(log_data)
    wcss_KMedoids.append(clusters_KMedoids.inertia_)

plt.plot(range(2, 11), wcss_KMedoids)
plt.title('Elbow Method (KMedoids)')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')

from kneed import KneeLocator
kl = KneeLocator(range(2, 11), wcss_KMedoids, curve="convex", direction="decreasing")
elbow_KMedoids = kl.elbow

# Highlight the elbow point on the plot
plt.axvline(x=elbow_KMedoids, linestyle='--', color='r', label=f'Elbow at {elbow_KMedoids} clusters')
plt.scatter(elbow_KMedoids, wcss_KMedoids[elbow_KMedoids - 2], color='red', s=100, zorder=5)  
plt.legend()
plt.tight_layout()
plt.show()
print(f"The elbow point is at {elbow_KMedoids} clusters.")

In [ ]:
# Hier
from sklearn.cluster import AgglomerativeClustering
from tslearn.metrics import cdist_dtw, dtw
from scipy.spatial.distance import cdist
from tslearn.barycenters import dtw_barycenter_averaging
import numpy as np
import matplotlib.pyplot as plt
from kneed import KneeLocator
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from tslearn.clustering import silhouette_score as ts_silhouette


n_clusters = 4
model_Agg = AgglomerativeClustering(n_clusters=n_clusters, metric='euclidean', linkage='ward')
labels_Agg = model_Agg.fit_predict(log_data)
print(labels_Agg)

cluster_centers_Agg = []
for label in np.unique(labels_Agg):
    center = log_data[labels_Agg == label].mean(axis=0)
    cluster_centers_Agg.append(center)
cluster_centers_Agg = np.array(cluster_centers_Agg)
uniq = np.unique(labels_Agg)

# Plot mean cluster centers as one plot
plt.figure(figsize=(10, 6))
for i, center in enumerate(cluster_centers_Agg):
    plt.plot(freqs, center, label=f'Cluster {uniq[i]} Center')
plt.xlabel("Frequency (Hz)")
plt.ylabel("Log(PSD)")
plt.title("Mean Cluster Centers (Agglomerative Clustering)")
plt.legend()
plt.show()


cluster_centers_Agg_Medoid = np.zeros((n_clusters, log_data.shape[1]))
for i, lab in enumerate(uniq):
    cluster_points = log_data[labels_Agg == lab]
    distances = cdist(cluster_points, cluster_points, metric='euclidean')
    medoid_idx = np.argmin(distances.sum(axis=1))
    cluster_centers_Agg_Medoid[i] = cluster_points[medoid_idx]

# Plot medoid centers as one combined plot
plt.figure(figsize=(10, 6))
for i, center in enumerate(cluster_centers_Agg_Medoid):
    plt.plot(freqs, center, label=f'Cluster {uniq[i]} Medoid')
plt.xlabel("Frequency (Hz)")
plt.ylabel("Log(PSD)")
plt.title("Medoid Cluster Centers (Agglomerative Clustering - Euclidean)")
plt.legend()
plt.tight_layout()
plt.show()

# Similarity
num_comparisons = n_clusters * (n_clusters - 1) // 2
dtw_distances_Medoid = np.zeros(num_comparisons)
dtw_distances = np.zeros(num_comparisons)

k = 0
for i in range(n_clusters):
    for j in range(i + 1, n_clusters):
        # Calculate DTW distance between centroids i and j
        dtw_distances_Medoid[k] = dtw(cluster_centers_Agg_Medoid[i], cluster_centers_Agg_Medoid[j])
        dtw_distances[k] = dtw(cluster_centers_Agg[i], cluster_centers_Agg[j])
        print(f"DTW distance between centroid {i} and centroid {j}: {dtw_distances[k]:.4f}")
        k += 1

# Check DTW distances against the threshold and print only if the condition is met
dtw_similarity_threshold = 3.5
simi_Medoid=np.any(dtw_distances_Medoid < dtw_similarity_threshold)
simi=np.any(dtw_distances < dtw_similarity_threshold)

if simi or simi_Medoid:
    print(f"Warning: Some centroids are too similar in shape (DTW distance < {dtw_similarity_threshold}).")
else:
    print("Centroids are sufficiently different based on DTW distance.")


sil = silhouette_score(log_data, labels_Agg)
print(f"Silhouette Score (Hierarchal): {sil: .2f}")
db_index = davies_bouldin_score(log_data, labels_Agg)
print(f"Davies-Bouldin Index (Hierarchal): {db_index: .2f}")
ch_score = calinski_harabasz_score(log_data, labels_Agg)
print(f"Calinski-Harabasz Index (Hierarchal): {ch_score: .2f}")

# Hier elbow
wcss_Agg = []
cluster_range = range(2, 11)

for i in range(2, 11):
    model_Agg = AgglomerativeClustering(n_clusters=i, metric='euclidean', linkage='ward')
    labels_Agg = model_Agg.fit_predict(log_data)
    
    # Calculate the WCSS for each cluster by summing the squared distances from each point to the cluster mean
    wcss = 0
    for label in np.unique(labels_Agg):
        cluster_points = log_data[labels_Agg == label]
        cluster_center = cluster_points.mean(axis=0) 
        distances = np.sum((cluster_points - cluster_center) ** 2) 
        wcss += distances

    wcss_Agg.append(wcss)

# Plot the elbow method
#plt.figure(figsize=(8, 6))
plt.plot(list(cluster_range), wcss_Agg, marker='o')
plt.title('Elbow Method (Agglomerative Clustering)')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')

# Use KneeLocator to find the elbow point
kl = KneeLocator(list(cluster_range), wcss_Agg, curve="convex", direction="decreasing")
elbow_Agg = kl.elbow

# Highlight the elbow point on the plot
idx = list(cluster_range).index(elbow_Agg)
plt.axvline(x=elbow_Agg, linestyle='--', color='r', label=f'Elbow at {elbow_Agg} clusters')
plt.scatter(elbow_Agg, wcss_Agg[idx], color='red', s=100, zorder=5)
plt.legend()
plt.tight_layout()
plt.show()
print(f"The elbow point is at {elbow_Agg} clusters.")

In [ ]:
# Hier with DTW
from tslearn.metrics import dtw, cdist_dtw
from sklearn.cluster import AgglomerativeClustering
from tslearn.barycenters import dtw_barycenter_averaging
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

n_clusters = 4

# DTW-ready format
log_data_ts = log_data[:, :, None]
# Step 1: Pairwise DTW distance matrix
distance_matrix = cdist_dtw(log_data_ts)

# Step 2: Agglomerative clustering with precomputed DTW distances
model_Agg = AgglomerativeClustering(n_clusters=n_clusters, metric="precomputed", linkage="average")
labels_Agg = model_Agg.fit_predict(distance_matrix)
print("Cluster labels:", labels_Agg)
print(labels_Agg.tolist())

# Keep numeric label order
uniq = np.unique(labels_Agg)

# Step 3A: Pointwise means
cluster_centers_mean = np.array([
    log_data[labels_Agg == lab].mean(axis=0) for lab in uniq
])

# Step 3B: DTW barycenters
cluster_centers_dba = []
for lab in uniq:
    cluster_points = log_data_ts[labels_Agg == lab]
    barycenter = dtw_barycenter_averaging(cluster_points)
    cluster_centers_dba.append(barycenter[:, 0])
cluster_centers_dba = np.array(cluster_centers_dba)

# Step 3C: Medoids
cluster_centers_medoid = np.zeros((n_clusters, log_data.shape[1]))
for k, lab in enumerate(uniq):
    cluster_points = log_data_ts[labels_Agg == lab]
    distances = cdist_dtw(cluster_points)
    medoid_idx = np.argmin(distances.sum(axis=1))
    cluster_centers_medoid[k] = cluster_points[medoid_idx, :, 0]

# Plot 1: pointwise means
plt.figure(figsize=(10, 6))
for k, center in enumerate(cluster_centers_mean):
    plt.plot(freqs, center, label=f'Cluster {uniq[k]} Mean')
plt.xlabel("Frequency (Hz)", fontsize=14)
plt.ylabel("Log(PSD)", fontsize=12)
plt.title("Pointwise Mean Centers (Agglomerative with DTW)", fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

# Plot 2: DTW barycenters
plt.figure(figsize=(10, 6))
for k, center in enumerate(cluster_centers_dba):
    plt.plot(freqs, center, label=f'Cluster {uniq[k]} DTW Barycenter')
plt.xlabel("Frequency (Hz)", fontsize=14)
plt.ylabel("Log(PSD)", fontsize=12)
plt.title("DTW Barycenters (Agglomerative with DTW)", fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

# Plot 3: medoids
plt.figure(figsize=(10, 6))
for k, center in enumerate(cluster_centers_medoid):
    plt.plot(freqs, center, label=f'Cluster {uniq[k]} Medoid')
plt.xlabel("Frequency (Hz)", fontsize=14)
plt.ylabel("Log(PSD)", fontsize=12)
plt.title("Medoid Centers (Agglomerative with DTW)", fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

# DTW similarity check for all 3 center definitions
num_comparisons = n_clusters * (n_clusters - 1) // 2
dtw_distances_means = np.zeros(num_comparisons)
dtw_distances_dba = np.zeros(num_comparisons)
dtw_distances_medoids = np.zeros(num_comparisons)

pair_names = []

k = 0
for i in range(n_clusters):
    for j in range(i + 1, n_clusters):
        pair_names.append(f"dtw{i}{j}")
        
        dtw_distances_means[k] = dtw(cluster_centers_mean[i], cluster_centers_mean[j])
        dtw_distances_dba[k] = dtw(cluster_centers_dba[i], cluster_centers_dba[j])
        dtw_distances_medoids[k] = dtw(cluster_centers_medoid[i], cluster_centers_medoid[j])
        
        k += 1

# Print compact summary
print(pair_names)
print("DTW(mean)       =", np.round(dtw_distances_means, 4))
print("DTW(barycenter) =", np.round(dtw_distances_dba, 4))
print("DTW(medoid)     =", np.round(dtw_distances_medoids, 4))



dtw_similarity_threshold = 3.5
if (np.any(dtw_distances_means < dtw_similarity_threshold) or np.any(dtw_distances_dba < dtw_similarity_threshold) or np.any(dtw_distances_medoids < dtw_similarity_threshold)):
    print(f"Warning: Some centers are too similar (DTW < {dtw_similarity_threshold}).")
else:
    print("Centers are sufficiently different based on DTW distance.")

# Metrics
sil = silhouette_score(distance_matrix, labels_Agg, metric="precomputed")
print(f"Silhouette Score (Hierarchical with DTW, precomputed): {sil:.2f}")
db_index = davies_bouldin_score(log_data, labels_Agg)
print(f"Davies-Bouldin Index (Hierarchical with DTW): {db_index:.2f}")
ch_score = calinski_harabasz_score(log_data, labels_Agg)
print(f"Calinski-Harabasz Index (Hierarchical with DTW): {ch_score:.2f}")